#Generación de Caracteres basados en Tensorflow RNN

In [ ]:
# RNN a nivel de caracteres con TensorFlow 2.x
# Asegúrate de tener TensorFlow instalado en Colab
import tensorflow as tf
import numpy as np
import os
import time


In [ ]:
# Subir archivo de texto
from google.colab import files
uploaded = files.upload()

# Cargar y preprocesar texto
file_path = list(uploaded.keys())[0]
text = open(file_path, 'r').read()
vocab = sorted(set(text))
print(f'{len(vocab)} caracteres únicos.')


In [ ]:
# Mapear caracteres a números
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)
text_as_int = np.array([char2idx[c] for c in text])


In [ ]:
# Longitud de secuencia para entrenamiento
seq_length = 20
examples_per_epoch = len(text) // seq_length

# Crear dataset con tf.data
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

# Crear pares (input, target)
def split_input_target(chunk):
  input_text = chunk[:-1]
  target_text = chunk[1:]
  return input_text, target_text

dataset = sequences.map(split_input_target)


In [ ]:
# Tamaño del batch
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
)


In [ ]:
# Hiperparámetros
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
  inputs = tf.keras.Input(batch_shape=(batch_size, None))  # Define input shape aquí
  x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(inputs)
  x = tf.keras.layers.SimpleRNN(
      rnn_units,
      return_sequences=True,
      stateful=True,
      recurrent_initializer='glorot_uniform'
  )(x)
  outputs = tf.keras.layers.Dense(vocab_size)(x)

  model = tf.keras.Model(inputs, outputs)
  return model


In [ ]:
# Construir el modelo
model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)

In [ ]:
# Función de pérdida
def loss(labels, logits):
  return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

# Compilar el modelo
model.compile(optimizer='adam', loss=loss)


In [ ]:
# Entrenar el modelo
EPOCHS = 10

history = model.fit(dataset, epochs=EPOCHS)


In [ ]:
# Guardar el modelo entrenado
model.save('char_rnn_model_tf.keras')


num_generate = 30: Generará 30 caracteres.

input_eval = [char2idx[s] for s in start_string]: Convierte el texto inicial a índices.

model.reset_states(): Limpia el estado de la RNN (relevante si stateful=True).

En cada paso:

El modelo predice la siguiente distribución de probabilidad.

Se ajusta por la temperature.

Se elige el siguiente carácter (índice) con tf.random.categorical.

Se actualiza el input para el siguiente paso.

Devuelve el texto generado concatenando el start_string + lo generado.

In [ ]:
# Cargar modelo para inferencia (batch_size=1)
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)
model.load_weights('char_rnn_model_tf.keras')
model.build(tf.TensorShape([1, None]))

def generate_text(model, start_string, num_generate=500, temperature=1.0):
  input_eval = [char2idx[s] for s in start_string]
  input_eval = tf.expand_dims(input_eval, 0)
  text_generated = []

  # Resetear el estado de la capa RNN
  model.layers[2].reset_states()

  for _ in range(num_generate):
    predictions = model(input_eval)
    predictions = tf.squeeze(predictions, 0)
    predictions = predictions / temperature
    predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

    input_eval = tf.expand_dims([predicted_id], 0)
    text_generated.append(idx2char[predicted_id])

  return start_string + ''.join(text_generated)

# Ejemplo
print(generate_text(model, start_string=u"Había una vez "))


In [ ]:
print(sorted(char2idx.keys()))
